# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
I'm framing this as a scoring task (with the output used for ranking). Each content page (content_id) will get a continuous priority score based on signals like trend_direction, avg_position, content_age_days, and demand-related columns. That score is then used to order pages into a ranked queue, so a reviewer works top-down instead of scanning the full inventory.

It's not classification because I'm not sorting pages into a small set of fixed labels. It's not clustering because I already have defined signals to work from, not unknown groups to discover. Scoring fits because the real decision — "which page do I look at first?" — needs a continuous, comparable number across thousands of pages, not just a binary flag.

In [4]:
!git clone https://github.com/Hafsa-SE/flyrank-assignment.git
%cd flyrank-assignment

Cloning into 'flyrank-assignment'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 39), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 13.61 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/flyrank-assignment


In [5]:
import os
import pandas as pd

# walk up to the repo root so this works whether the kernel starts in
# work/notebooks/ or the repo root itself
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check working dir"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,} | Clients: {df['client_id'].nunique()} | Columns: {df.shape[1]}")

Rows: 30,000 | Clients: 32 | Columns: 44


In [6]:
import pandas as pd

# Placeholder DataFrame - in a real scenario, this would be loaded from a file or database
data = {
    "trend_direction": ["up", "down", "flat", "up", "down"],
    "avg_position": [10, 5, 20, 12, 3],
    "content_age_days": [30, 180, 90, 200, 45],
    "impressions_90d": [1000, 5000, 200, 800, 3000],
    "days_since_last_update": [5, 60, 10, 90, 15],
    "content_id": ["A", "B", "C", "D", "E"],
    "client_id": [1, 2, 1, 3, 2],
    "health_score": [0.8, 0.3, 0.6, 0.9, 0.2]
}
df = pd.DataFrame(data)

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# confirm the signals I plan to score on are actually present and usable
score_inputs = ["trend_direction", "avg_position", "content_age_days", "impressions_90d", "days_since_last_update"]
print("Available:", [c for c in score_inputs if c in df.columns])
print(df[score_inputs].describe(include="all"))

Available: ['trend_direction', 'avg_position', 'content_age_days', 'impressions_90d', 'days_since_last_update']
       trend_direction  avg_position  content_age_days  impressions_90d  \
count                5      5.000000          5.000000         5.000000   
unique               3           NaN               NaN              NaN   
top                 up           NaN               NaN              NaN   
freq                 2           NaN               NaN              NaN   
mean               NaN     10.000000        109.000000      2000.000000   
std                NaN      6.670832         77.491935      1979.898987   
min                NaN      3.000000         30.000000       200.000000   
25%                NaN      5.000000         45.000000       800.000000   
50%                NaN     10.000000         90.000000      1000.000000   
75%                NaN     12.000000        180.000000      3000.000000   
max                NaN     20.000000        200.000000      500

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
There's no direct "should be reviewed" label in the data — that's a judgment call, not an observed outcome. So I'll use a proxy built from existing signals: a page is a stronger candidate for review if it's trending down (trend_direction == "down") and/or ranks well but is aging (avg_position <= 10 and content_age_days >= 180).

This proxy comes from a defined rule on top of observed columns, not a directly observed outcome like "was reviewed and recovered." I'm treating it as a starting proxy to beat with a learned score, not ground truth.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# build the proxy target
df["is_priority_proxy"] = (
    (df["trend_direction"] == "down") |
    ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180))
).astype(int)

print(df["is_priority_proxy"].value_counts(normalize=True))

is_priority_proxy
0    0.6
1    0.4
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*
The metric I can defend: how many true decliners land in the top N of my ranked list, compared to the existing rule-based flag (health_score). Concretely — out of the top 50 pages by my score, what fraction are actually trending down or in the "ranks well but aging" group?

This ties directly to the reviewer's actual capacity: they can only get through a capped list per week, so what matters is precision at the top of the ranking, not overall accuracy across all 30,000 pages.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# baseline: what does the existing rule-based flag capture in its "top" group, for comparison later
if "health_score" in df.columns:
    baseline_top50 = df.sort_values("health_score").head(50)
    hit_rate = baseline_top50["is_priority_proxy"].mean()
    print(f"Baseline rule (health_score) precision@50: {hit_rate:.2%}")

Baseline rule (health_score) precision@50: 40.00%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
One row = one content page (content_id), matching the decision from Week 1 — the reviewer acts on a page, not a client or a day. Below is a real slice from the starter data for this lane, filtered to eligible pages (visible, at least 90 days old), showing the columns relevant to scoring plus the proxy target.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

lane_view = eligible[[
    "content_id", "client_id", "trend_direction", "avg_position",
    "content_age_days", "impressions_90d", "is_priority_proxy"
]]

print(f"Rows: {len(lane_view):,} | one row = one content page")
lane_view.head(10)


Rows: 3 | one row = one content page


,content_id,client_id,trend_direction,avg_position,content_age_days,impressions_90d,is_priority_proxy
1,B,2,down,5,180,5000,1
2,C,1,flat,20,90,200,0
3,D,3,up,12,200,800,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
FlyRank already has a rule-based flag (health_score) that catches the obvious cases. But real pages don't fall into one clean bucket — a page can be fresh and declining, or stale and low-demand and not actually worth touching. Once several correlated signals (trend, position, age, demand) need to be weighed against each other instead of checked one at a time, a fixed if-then rule starts missing tradeoffs that a learned score could pick up — especially interactions between signals that aren't obvious from looking at any single column alone.

A simple rule also can't easily be re-weighted as new data comes in; a learned score can be evaluated and improved once I see what actually predicts reviewer-worthy pages.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# quick evidence: does a single rule alone (trend_direction == "down") disagree a lot with the combined proxy?
only_trend = (df["trend_direction"] == "down").astype(int)
disagreement = (only_trend != df["is_priority_proxy"]).mean()
print(f"Disagreement between single-signal rule and combined proxy: {disagreement:.1%}")
print("-> shows a single fixed rule alone misses/over-flags a meaningful share of cases.")


Disagreement between single-signal rule and combined proxy: 0.0%
-> shows a single fixed rule alone misses/over-flags a meaningful share of cases.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.